# 06 · Making the application trustworthy

**Session 01 · Building AI applications for research and data · notebook 6 of 6**

## Objective

Notebook 05 ended on a question: what if every number in an answer carried its label, and a program checked each number against the record it names? The answer to the main question holds numbers from a tool, from a passage and from the model's own arithmetic, in plain text that does not say which is which. This notebook builds that check, and it is the last layer of trust. Retrieval found the right evidence (03), a tool returned exact figures (04), and a step limit and a trace bounded the agent (05). Now the numbers themselves are checked.

The protocol is **Proof-Carrying Numbers** (PCN), the program's approach to trustworthy data ([paper](https://arxiv.org/abs/2509.06902), [code](https://github.com/worldbank/pcn)): the model labels each number with the record it comes from, and a verifier checks each number against exactly that record. Then the notebook asks a second question, about the whole application and not one answer: how do we know that it behaves well across many questions, and that a change made it better and not worse? It answers with an evaluation suite, a fixed set of questions with known right evidence, run on two versions of the application and compared.

Four terms, in plain words:

- A **claim tag** is a label that the model writes around a number: it names the record the number comes from, and how exactly to compare the two. The tag is the `claim_id` of notebook 05 with a little more.
- A **verifier** reads the tags and checks each number against its record. The one here is `pcn-core`, the Python core of the PCN protocol. Each number gets a mark: **verified** (it matches its record), **flagged** (it does not) or **unmarked** (it has no tag, so there is nothing to check it against).
- An **evaluation suite** is a fixed list of questions, each with the evidence and the behaviour that a good answer needs. The application runs on every question, and a table shows how it did.
- A **configuration** is one version of the application. The suite compares two.

Four kinds of check, and where each has appeared:

| Check | Question it answers | Where |
|---|---|---|
| Retrieval | Did the application find the right evidence? | Notebook 03, section 6 |
| Grounding | Is the answer written from that evidence? | Citations in the answer |
| Verification | Does each number match its source record? | Claim tags and `pcn-core`, this notebook |
| Evaluation | Does the application behave across many questions? | The suite, this notebook |

## The application after this notebook

![Diagram: the question goes to the agent with tools of notebook 05, which writes an answer with claim tags; the verifier, pcn-core, puts a mark on each number: verified, flagged or unmarked. Separately, the evaluation suite of eight questions produces a table per question for each configuration of the application.]()

## What you will see

Five short scenes:

1. A claim tag, step by step, and the model's answer to the main question with every number checked.
2. A wrong number and a number with no tag: how the verifier marks them, and how a chatbot shows those marks to a reader.
3. Two versions of the application, to be compared.
4. The evaluation suite: the eight questions, and a table of results for each version.
5. How to read the table, and what the suite does not cover.

## How to run it

Run the cells in order. Without an API key, the model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. With a key in `WORKSHOP_API_KEY`, the same cells call the model live, and the answers will differ in wording from the recorded ones. The searches, the tool results, the checks and the tables are computed when the cell runs. The exercises at the end run without a key.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
import json
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

docs = w.load_documents()
store = w.build_store(w.split_sections(docs))   # the store of notebooks 02 and 03
search_documents = w.make_search_tool(store, k=3, since="2020")   # the tools of notebooks 04 and 05
tools = [search_documents, w.get_indicator]

INSTRUCTIONS_CLAIMS = """You are a research assistant for development data.
Use get_indicator for every statistic and search_documents for definitions and context. Do not quote figures from memory.
Write every number that comes from get_indicator inside a claim tag that names its claim_id, like
<claim id="IT.NET.USER.ZS:KEN:2024" policy="rounded" decimals="1">35.0</claim>.
Name the indicator and the source. If the tools do not provide what is needed, say so.
If the question is ambiguous, ask which reading is meant."""

## 1. Claim tags

The instruction in the cell above asks the model to wrap each figure from the tool in a tag. The picture shows the parts of one tag, and what the verifier does with it: it takes the record named by the `claim_id`, compares it with the number inside the tag by the stated policy (rounded to some decimals, or exact), and marks the result. This is the Proof-Carrying Numbers protocol of the program, and `pcn-core` is its comparison code.

![Diagram: a claim tag wraps a number and carries the claim_id, which names the indicator, the country and the year, and a policy that says how to compare, here rounded to one decimal. The verifier takes the record, 34.9762, rounds it to one decimal, gets 35.0, finds it equal to the number shown, and marks it verified.]()

The next cell gives the model the four series that the main question needs and runs it. What comes back is drawn the way a chatbot would draw it: the answer as a reader sees it, with a small mark after each number that the application could check. Hover over a mark to see why it is there: which record the model named, where the tool got it, and how the two were compared. The labels themselves are folded away under the answer, for anyone who wants to see what the model really wrote.

In [3]:
results = [w.indicator_lookup(i, c, start=2010) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
observations = w.observations_from_results(results)   # claim_id -> the value the tool returned

w.use("nb06_claims_main")
reply = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
w.show_chat(w.MAIN_QUESTION, reply.content, observations, results, raw=True)

Every number carries a ✓. Behind each mark is a check that the application made on its own: the model named a record, the application looked the record up in what the tool returned, and compared the two by the rule in the label. The table lists those checks, one row per number: the record each label named, what the record holds, and the mark.

In [4]:
w.show_checks(w.verify(reply.content, observations))

number,claim_id,the record holds,mark
35.0,IT.NET.USER.ZS:KEN:2024,34.9762,verified
31.7,IT.NET.USER.ZS:RWA:2024,31.7033,verified
7.2,IT.NET.USER.ZS:KEN:2010,7.2,verified
8.0,IT.NET.USER.ZS:RWA:2010,8,verified
"2,133",NY.GDP.PCAP.CD:KEN:2024,2133.46,verified
"1,060",NY.GDP.PCAP.CD:RWA:2024,1059.94,verified
"1,092",NY.GDP.PCAP.CD:KEN:2010,1091.52,verified
587,NY.GDP.PCAP.CD:RWA:2010,586.6,verified


Why not simply look for a tool value equal to each number, without labels? Because a match by value cannot tell a source from a coincidence: "28", a subtraction the model did itself, equals Rwanda's 2022 value of 28.0, and a value-matching check would call that a source. With a label, the verifier reads which record the model meant instead of guessing. Years are not figures to check, so they are skipped. The application can now check every number that the model says comes from the tool.

The picture and the cell above follow the program's JavaScript packages for the web, `@pcn-js/core` and `@pcn-js/ui` ([`worldbank/pcn`](https://github.com/worldbank/pcn)), whose `ClaimMark` component draws the ✓ and the hover in a real chatbot. The repository has a small demo chat app that uses it. The notebook imitates that view with plain HTML, and the marks come from the Python verifier.

## 2. A wrong number, and a number with no label

Two things can go wrong, and the cell shows one of each. In this recorded answer, one labelled figure has been altered (Rwanda's internet use is given as 35.7 instead of 31.7), and one figure has no label because it did not come from the data: "about 70 percent … use mobile money". The verifier flags the first with a ⚠. The second it marks **unmarked**: it carries no claim, so there is nothing to check it against, and no mark is drawn, whether or not it happens to be right.

In [5]:
w.use("nb06_claims_error")
flawed = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
w.show_chat(w.MAIN_QUESTION, flawed.content, observations, results)
w.show_checks(w.verify(flawed.content, observations))

number,claim_id,the record holds,mark
35.0,IT.NET.USER.ZS:KEN:2024,34.9762,verified
35.7,IT.NET.USER.ZS:RWA:2024,31.7033; shown 35.7,flagged
"2,133",NY.GDP.PCAP.CD:KEN:2024,2133.46,verified
"1,060",NY.GDP.PCAP.CD:RWA:2024,1059.94,verified
70,no tag,none,unmarked


Hover over the ⚠: the tooltip shows the record the model named, the figure the tool returned, and that the two are not equal. "About 70 percent" carries nothing. That absence is part of the design. The protocol's own principle reads: *trust is earned only by proof, and the absence of a verification mark communicates uncertainty.* The same holds for the numbers of notebook 05 that the model worked out itself, such as a rise of "about 28 percentage points". They have no single record, so they stay unmarked, and a reader knows to look at them.

What the check says nothing about: whether the record is the right one for the question, whether the method behind it is sound, or whether the sentence around the number interprets it correctly. Those stay with the reader. The application can now flag a number that does not match its record, and leave unmarked a number that has no record.

## 3. Two versions of the application

A check on one answer is not yet a measure of the application. The second half of the notebook compares two configurations on the same questions:

| Configuration | What it uses | Built in |
|---|---|---|
| **A · retrieval only** | The search, then the model writes from the passages. No tools, no tags. | Notebook 03 |
| **B · tools and claims** | The agent with both tools, the claim instructions above, and the verifier. | Notebooks 04 and 05, and this one |

The cell defines both as functions that take a question and return the answer, the passages the search found, and the records the tools returned.

In [6]:
INSTRUCTIONS_RAG = """You are a research assistant for development data.
Answer only from the supplied passages and cite them in the square-bracket form given. If they do not contain what is needed, say so."""
rag_prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS_RAG), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])
rag_chain = rag_prompt | model
agent = create_agent(model, tools, system_prompt=INSTRUCTIONS_CLAIMS)


def app_rag(question, scenario):
    hits = store.similarity_search(question, k=4)
    w.use(scenario)
    reply = rag_chain.invoke({"context": w.format_context(hits), "question": question})
    return reply.content, hits, {}


def app_tools(question, scenario):
    w.use(scenario)
    result = agent.invoke({"messages": [HumanMessage(content=question)]}, config={"recursion_limit": 10})
    obs = w.observations_from_results(w.tool_results(result, "get_indicator"))
    hits = store.similarity_search(question, k=4)  # for the retrieval column; the agent may or may not have searched
    return w.final_text(result), hits, obs

The application can now be run as a function of a question, which is what a suite needs.

## 4. The evaluation suite

The suite is eight questions from `assets/questions.json`, each of a different kind: the main question, factual ones, an ambiguous one, a missing year, an out-of-scope one, a definition, and one where the evidence is outdated. Each has the documents and numbers a good answer should use, and the behaviour it should show.

In [7]:
questions = w.load_questions()
w.show_table(["id", "kind", "question", "what a good answer does"], [(q["id"], q["category"], q["question"], q["behaviour"]) for q in questions])

id,kind,question,what a good answer does
q1,main,"How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.","Reports the latest year for each indicator and country, names the indicators and the source, and compares levels and trends."
q2,factual,What share of Rwanda's population used the internet in 2023?,Gives 28.8 percent with the indicator name and the source.
q3,ambiguous,How connected is Kenya?,"Asks which measure is meant, or states the indicator it chose before answering, and distinguishes internet use from mobile subscriptions."
q4,missing-data,What share of Kenya's population used the internet in 2025?,"States that no 2025 observation is available and gives the latest available year, 2024, at 35.0 percent."
q5,out-of-scope,How have Kenya's coffee export earnings changed in recent years?,States that the available documents describe the sector without figures and that the data tools hold no coffee-export indicator; gives no number.
q6,factual,Which of the two countries had faster GDP growth in 2023?,"Rwanda, 8.6 percent against 5.7 percent, with the source."
q7,definition,"How is the internet-users indicator defined, and what should be kept in mind when comparing years?","Gives the three-month, any-device definition from the ITU and at least one caveat on comparability."
q8,outdated-evidence,What are the latest available figures for internet use in Kenya and Rwanda?,"Gives the 2024 values, 35.0 and 31.7 percent. Quoting 2015 values from the 2016 summary is a failure."


For each question and each configuration, the table reports four things:

- **retrieval**: did the four passages that the search found include an expected document?
- **numbers**: how many of the expected numbers appear in the answer?
- **cited**: does the answer cite a retrieved document, or name a source?
- **verified / flagged / unmarked**: the verifier's counts (configuration B only, since only B uses tags).

In [8]:
def evaluate(app, prefix):
    rows = []
    for qd in questions:
        answer, hits, obs = app(qd["question"], f"{prefix}_{qd['id']}")
        found, total = w.numbers_present(answer, qd["expected_numbers"])
        cited = w.grounded(answer, hits) or "WDI" in answer or "World Development Indicators" in answer
        if obs:
            checks = w.verify(answer, obs)
            v, f, u = (sum(c.status == s for c in checks) for s in ("verified", "flagged", "unmarked"))
            ver = f"{v} / {f} / {u}"
        else:
            ver = "not applicable" if prefix.endswith("rag") else "no tool data used"
        rows.append(w.EvalRow(qd["id"], qd["category"], "hit" if w.retrieval_hit(hits, qd["expected_documents"]) else "miss", f"{found}/{total}" if total else "none expected", "yes" if cited else "no", ver))
    return rows


print("A · retrieval only\n")
w.show_eval(evaluate(app_rag, "nb06_rag"))
print("\nB · tools and claims\n")
w.show_eval(evaluate(app_tools, "nb06_tools"))

A · retrieval only



id,category,retrieval,numbers,cited,verified / flagged / unmarked
q1,main,hit,4/4,yes,not applicable
q2,factual,hit,1/1,yes,not applicable
q3,ambiguous,hit,none expected,yes,not applicable
q4,missing-data,hit,none expected,yes,not applicable
q5,out-of-scope,hit,none expected,yes,not applicable
q6,factual,hit,2/2,yes,not applicable
q7,definition,hit,none expected,yes,not applicable
q8,outdated-evidence,hit,0/2,yes,not applicable



B · tools and claims



id,category,retrieval,numbers,cited,verified / flagged / unmarked
q1,main,hit,4/4,yes,4 / 0 / 0
q2,factual,hit,1/1,yes,1 / 0 / 0
q3,ambiguous,hit,none expected,yes,no tool data used
q4,missing-data,hit,none expected,yes,1 / 0 / 0
q5,out-of-scope,hit,none expected,yes,no tool data used
q6,factual,hit,2/2,yes,2 / 0 / 0
q7,definition,hit,none expected,yes,no tool data used
q8,outdated-evidence,hit,2/2,yes,2 / 0 / 0


The application can now be measured across questions, and two versions of it compared.

## 5. Reading the table

- **q8, outdated evidence.** Both configurations "hit" on retrieval. Yet A answers from the 2016 summary, with 2015 figures, and finds none of the two expected numbers, while B fetches the series and reports 2024. No single column tells the whole story: retrieval alone said A was fine.
- **q3, ambiguous.** Neither configuration should answer with a single number. B asks which measure is meant, as its instructions say. A answers without asking, with both measures at once. The numbers column shows "none expected" for both: the behaviour has to be read, not counted.
- **q4, missing data.** Both say that 2025 is unavailable and give 2024. B's statement rests on an empty tool result.
- **q5, out of scope.** Both give no number. A's retrieval "hit" is the coffee note, which is the expected document and contains no figures.
- **Verified, flagged, unmarked.** In B every number that came from the data tool is verified, and nothing is flagged. In A there is nothing to verify against, because no number came from a tool.

A table like this is what separates a demonstration from a measured application. It is rerun after every change to the instructions, the tools, the store or the model. For suites at scale, with a model as a judge for the behaviours that a rule cannot check, see the program's evaluation cookbook; libraries such as Ragas and DeepEval provide those judges.

## 6. What the suite does not cover

- Whether the application chose the right indicator for the question. A verified number from the wrong series is verified.
- Whether the interpretation around the numbers is right.
- Behaviour on questions that nobody wrote down. The suite grows from use: every failure found in use becomes a question.

## Key points

- Notebook 05 could only match numbers to values, and guessed. A claim tag names the record that a number comes from, and the verifier compares the two: verified, flagged, or unmarked.
- The rule is that an untagged number gets no mark. The verifier cannot say whether a number is right, only whether it matches the record it names.
- Evaluation runs the application over questions with known evidence and expected behaviour, and is rerun on every change. One column is rarely enough: on q8 retrieval alone said the weaker version was fine.
- The two configurations differ most on outdated evidence and on numbers: tools make numbers checkable, and retrieval alone does not.

## Exercise

All of these run without a key, on the answer of section 1 (`reply`).

1. Take the answer of section 1 and change one number inside a tag: `tampered = reply.content.replace(">35.0<", ">35.7<")`, then `w.show_checks(w.verify(tampered, observations))`. Which mark changes, and what does the record hold?
2. Remove all the tags: `plain = w.strip_claims(reply.content)`, then `w.show_checks(w.verify(plain, observations))`. What happens to every number, and why is it not a "verified" result?
3. Change the policy of one tag from rounded to exact: `strict = reply.content.replace('policy="rounded" decimals="1"', 'policy="exact"', 1)`, then verify it. Why does 35.0 stop matching a record of 34.9762, and which policy would you want for a published percentage?
4. Look at the eight questions of section 4. Which kind of question would you add next, and what would a good answer to it do? Failures found in use are the best source.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [07 · Lab: chat with your documents](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/07-lab-chat-with-your-documents.ipynb)